In [61]:
import pandas as pd
import numpy as np
import re
import ftfy

OLD = r'C:\Users\USER\Desktop\Projects\philosophy_and_popculture'
NEW = r'C:\Users\USER\Desktop\Projects\philosophy_vs_popculture_clean'

# ── 1. LOAD ALL SOURCES ────────────────────────────────────────────────

def load_tagged(path, platform, source_type, text_col='comment'):
    try:
        df = pd.read_csv(path, usecols=[text_col], on_bad_lines='skip')
        df = df.rename(columns={text_col: 'comment'})
        df['platform'] = platform
        df['source_type'] = source_type
        print(f"  ✓ {platform:12s}: {len(df):>8,} rows")
        return df
    except Exception as e:
        print(f"  ✗ {platform}: {e}")
        return pd.DataFrame()

print("Loading sources...")

# Philosophy — from the labeled raw file
raw = pd.read_csv(f'{NEW}/data/final/philosophy_vs_popculture.csv')
philo_df = raw[raw['source_type'] == 'philosophy'][['comment', 'platform', 'source_type']].copy()
print(f"  ✓ {'philosophy':12s}: {len(philo_df):>8,} rows")

# Pop culture — correct paths
reddit    = load_tagged(f'{OLD}/data/pop_texts/merged/reddit_comments_merged.csv',   'reddit',    'pop_culture')
youtube   = load_tagged(f'{OLD}/data/pop_texts/merged/youtube_comments_merged.csv',  'youtube',   'pop_culture')
instagram = load_tagged(f'{OLD}/data/pop_texts/merged/instagram_comments_merged.csv','instagram', 'pop_culture')
tiktok    = load_tagged(f'{OLD}/data/pop_texts/merged/tiktok_comments_merged.csv',   'tiktok',    'pop_culture')

df = pd.concat([philo_df, reddit, youtube, instagram, tiktok], ignore_index=True)
print(f"\nCombined raw: {df.shape}")
print(df['platform'].value_counts())

Loading sources...
  ✓ philosophy  :   16,738 rows
  ✓ reddit      :   29,972 rows
  ✓ youtube     :   62,453 rows
  ✓ instagram   :    9,993 rows
  ✓ tiktok      :    2,114 rows

Combined raw: (121270, 3)
platform
youtube       62453
reddit        29972
philosophy    16738
instagram      9993
tiktok         2114
Name: count, dtype: int64


In [62]:
# ── 2. CLEAN ───────────────────────────────────────────────────────────

def clean_text(text):
    if not isinstance(text, str):
        return None
    text = ftfy.fix_text(text)
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 0 else None

print("Cleaning text...")
df['comment'] = df['comment'].apply(clean_text)
df = df.dropna(subset=['comment'])
print(f"After cleaning: {df.shape}")

Cleaning text...
After cleaning: (115315, 3)


In [63]:
# ── 3. FILTER ──────────────────────────────────────────────────────────

# Minimum length
df = df[df['comment'].str.len() >= 30]

# Book/scraper artifacts
artifact_patterns = [
    r'^[\d\s\.\,]+$',
    r'^page \d+',
    r'oceanofpdf',
    r'get any book for free',
    r'^\[deleted\]$',
    r'^\[removed\]$',
]
mask = pd.Series([True] * len(df), index=df.index)
for pat in artifact_patterns:
    mask = mask & ~df['comment'].str.contains(pat, case=False, na=False)
df = df[mask]

# Deduplicate within source type
df = df.drop_duplicates(subset=['comment', 'source_type'])

print(f"After filtering: {df.shape}")
print(df['source_type'].value_counts())
print()
print(df['platform'].value_counts())

After filtering: (94054, 3)
source_type
pop_culture    79988
philosophy     14066
Name: count, dtype: int64

platform
youtube       52094
reddit        20815
philosophy    14066
instagram      6018
tiktok         1061
Name: count, dtype: int64


In [64]:
# ── 4. RECHUNK PHILOSOPHY ──────────────────────────────────────────────

def rechunk_philosophy(df, window=150, step=75):
    philo = df[df['source_type'] == 'philosophy'].copy()
    pop   = df[df['source_type'] != 'philosophy'].copy()
    
    full_text = ' '.join(philo['comment'].tolist())
    words = full_text.split()
    
    chunks = []
    for i in range(0, len(words) - window, step):
        chunk = ' '.join(words[i:i+window])
        if len(chunk) > 50:
            chunks.append({
                'comment': chunk,
                'platform': 'philosophy',
                'source_type': 'philosophy'
            })
    
    philo_rechunked = pd.DataFrame(chunks)
    print(f"Philosophy: {len(philo):,} fragments → {len(philo_rechunked):,} chunks (~150 words each)")
    return pd.concat([philo_rechunked, pop], ignore_index=True)

df = rechunk_philosophy(df)
print(f"Final shape: {df.shape}")
print(df['source_type'].value_counts())

Philosophy: 14,066 fragments → 2,388 chunks (~150 words each)
Final shape: (82376, 3)
source_type
pop_culture    79988
philosophy      2388
Name: count, dtype: int64


In [65]:
# ── 5. SANITY CHECK ────────────────────────────────────────────────────

print("=== PHILOSOPHY SAMPLE ===")
for _, row in df[df['source_type']=='philosophy'].sample(5).iterrows():
    print(f"  {row['comment'][:250]}\n")

print("=== POP CULTURE SAMPLE ===")
for _, row in df[df['source_type']=='pop_culture'].sample(5).iterrows():
    print(f"  [{row['platform']}] {row['comment'][:250]}\n")

=== PHILOSOPHY SAMPLE ===
  sing it to my own ears. there are other singers, to be sure, whose voices are softened, whose hands are eloquent, whose eyes are expressive, whose hearts are awakened, only when the house is full: i am not one of them. he who will one day teach men t

  of delight were its refreshment; the blind man's life crept along on a staff. yet what happened to me? how did i free myself from disgust? who rejuvenated my eyes? how did i fly to the height where the rabble no did my disgust itself create wings and

  other points between them, however small the distance between them may be: every distance can be halved, and the halves can be halved again, and so on ad infinitum. in time, similarly, however little time may elapse between two moments, it seems evid

  of consciousness in the form of an exaggerated extraverted attitude which richly deserves weininger's description "misautic". inasmuch as the introverted attitude is based upon a universally present, extremely 

In [66]:
# ── 6. SAVE ────────────────────────────────────────────────────────────

out_path = f'{NEW}/data/final/philosophy_vs_popculture_v2.csv'
df[['comment', 'platform', 'source_type']].to_csv(out_path, index=False)

print(f"✅ Saved: {out_path}")
print(f"Shape: {df.shape}")
print(df['source_type'].value_counts())
print(df['platform'].value_counts())

✅ Saved: C:\Users\USER\Desktop\Projects\philosophy_vs_popculture_clean/data/final/philosophy_vs_popculture_v2.csv
Shape: (82376, 3)
source_type
pop_culture    79988
philosophy      2388
Name: count, dtype: int64
platform
youtube       52094
reddit        20815
instagram      6018
philosophy     2388
tiktok         1061
Name: count, dtype: int64
